# <span style="color:Thistle">Cómo Python representa los datos en la memoria</span>

#### <span style="color:beige">Sesión 1</span>
---

<span style="color:Gold">Antes de empezar:</span>

- Crea un nuevo proyecto en github para estas sesiones
- Crea y activa tu entorno virtual
- Instala numpy con `pip install numpy` (_La versión que he instalado es 2.4.2 (Febrero 2026)_)

---
Ahora sí, <span style="color:Gold">empezamos:</span>

Vamos a bajar al nivel de cómo python representa los datos en memoria, y cuando hablamso de memoria, hablamos de memoria RAM (_ahora que está tan cara y toda la industria del hardware desde móviles a consolas está temblando_).

`OJO! No confundir memoria con disco duro, es un fallo típico de gente no-técnica. Pero nosotros, los que hemos estudiado cualquier rama de la informática, cuando hablamos de memoria hablamos de memoria RAM.`

Para que los cálculos sean eficientes (en tiempo y recursos de CPU), necesitamos entender cómo se almacenan y manipulan los datos. Esta sección describe cómo se gestionan los arrays de datos en el propio lenguaje Python, y cómo NumPy mejora esto. <span style="color:Gold">Entender esta diferencia es fundamental para comprender **porqué** usamos NumPy.</span>

A los que usamos Python, a menudo nos han vendido su facilidad de uso, una de cuyas características es el **tipado dinámico**. 

<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿A qué nos referimos cuando hablamos de tipado dinámico?
</div>

Mientras que un lenguaje de tipado estático como C o Java requiere que cada variable sea declarada explícitamente, un lenguaje de tipado dinámico como Python omite esta especificación. Por ejemplo, en C podrías especificar una operación particular de la siguiente manera:

```c
/* Código C */
int result = 0;
for(int i=0; i<100; i++){
    result += i;
}
```
<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Qué hace este código?
</div>



En Python la operación equivalente se escribiría así:

In [ ]:
# Código Python
resultado = 0
for indice in range(100):
    resultado += indice

Observa la principal diferencia: en C, los tipos de datos de cada variable se declaran explícitamente, mientras que en Python los tipos se infieren de forma dinámica. 

<span style="color:Gold">Pero vamos, que esto no es nada nuevo a estas alturas del curso.</span> Como ya sabes, esto significa, que podemos asignar cualquier tipo de dato a cualquier variable:

In [ ]:
# Código Python
variable = 4
variable = "cuatro"

<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Quién decide cada variable qué tipo tiene?
</div>

Aquí hemos cambiado el contenido de `variable` de un entero a una cadena de texto. Lo mismo en C provocaría (dependiendo de la configuración del compilador) un error de compilación u otras consecuencias no deseadas:

```c
/* Código C */
int variable = 4;
variable = "cuatro";  // FALLA
```

Este tipo de flexibilidad es una de las razones por las que Python y otros lenguajes de tipado dinámico son <span style="color:Gold">MUY MUY cómodos</span> y fáciles de usar. Entender *cómo* funciona esto es una pieza importante para aprender a analizar datos de forma eficiente y efectiva con Python. Pero esta flexibilidad de tipos también implica que las variables de Python son más que solo su valor; también contienen información extra sobre el tipo del valor.

<font color='gold'>Quizá esto es algo que no te habias planteado, pero tiene implicaciones MUY grandes. </font>

<div class="alert alert-block alert-success">
<b>Ventajas:</b> <br>
- Curva de aprendizaje muy suave ya que no te van a dar la típica introducción de int vs. double vs. string que poca gente entiende el 1º día de clase.<br>
- El compilador no es estricto y eso accelera el desarrollo.<br>
- Muy cómodo.
</div>

<div class="alert alert-block alert-danger">
<b>Desventajas: </b><br>- Disminución de la eficiencia.<br>
-¿Hay más desventajas?
</div>

<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Se te ocurre alguna otra desventaja?
</div>




---
## Un entero en Python: Como se representa

La implementación estándar de Python está escrita en C. <span style="color:Gold">Esto significa que cada objeto de Python es simplemente una estructura C hábilmente disfrazada</span>, que contiene no solo su valor, sino también otra información. Por ejemplo, cuando definimos un entero en Python como `variable = 10000`, `variable` no es simplemente un entero «en bruto». Es en realidad un puntero a una estructura C compuesta, que contiene varios valores. Mirando el código fuente de Python 3.X, encontramos que la definición del tipo entero (`long`) tiene efectivamente este aspecto:

```c
struct _longobject {
    long ob_refcnt;
    PyTypeObject *ob_type;
    size_t ob_size;
    long ob_digit[1];
};
```

Un único entero en Python 3.X contiene en realidad cuatro componentes:

- `ob_refcnt`: 
    - <span style="color:Gold">Contador de referencias.</span>
    - Lleva la cuenta de cuántas referencias existen a este objeto.
    - Python usa esto para la gestión automática de memoria (garbage collection).
    - Cuando llega a 0, el objeto puede ser liberado.
- `ob_type`: 
    - Codifica el tipo de la variable.
    - Permite que Python sepa que este objeto es un entero.
    - Es lo que hace posible el sistema de tipos dinámicos de Python
- `ob_size`: 
    - Especifica el tamaño del dato.
    - Se refiere a los bits usados en memoria.
- `ob_digit`: 
    - Array flexible de datos.
    - Permite representar números de tamaño arbitrario.
    - Contiene el valor entero real del entero de la variable Python.


<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Cómo sabe el compilador que una variable no se está usando?
</div>

<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Dónde guardamos el tamaño REAL de la variable y su valor?
</div>

Esto significa que hay una <span style="color:Gold">cierta sobrecarga al almacenar un entero en Python en comparación con un entero en un lenguaje compilado como C</span>:

![Distribución de memoria de un entero](https://i.imgur.com/BczS9ws.png)

Aquí `PyObject_HEAD` es la parte de la estructura que contiene el contador de referencias, el código de tipo y otras piezas mencionadas anteriormente.

Observa la diferencia: <span style="color:Gold">un entero en C es esencialmente una etiqueta para una posición en memoria cuyos bytes codifican un valor entero. Un entero en Python es un puntero a una posición en memoria que contiene toda la información del objeto Python</span>, incluidos los bytes que contienen el valor entero. Esta información extra en la estructura del entero Python es lo que permite que Python sea tan libre y dinámico. Sin embargo, toda esta información adicional en los tipos de Python tiene un coste, que se vuelve especialmente evidente en estructuras que combinan muchos de estos objetos.


<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Qué coste tiene en memoria esta arquitectura de python?
</div>

<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Por qué los lenguajes de programación más comunes y los sistemas operativos están escritos en C?
</div>

---
## Una Lista en Python es más que una simple Lista

Vamos a ver qué ocurre cuando usamos una estructura de datos de Python que almacena muchos objetos. El contenedor mutable estándar de múltiples elementos en Python es la lista (`list`). 

<div class="alert alert-block alert-info">
<b>Pregunta:</b> ¿Qué significa mutable?
</div>

Podemos crear una lista de enteros de la siguiente manera:

In [ ]:
lista_enteros = list(range(10))
lista_enteros

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [ ]:
type(lista_enteros[0])

int

O, de forma similar, una lista de cadenas de texto:

In [ ]:
lista_cadenas = [str(numero) for numero in lista_enteros]
lista_cadenas

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

In [ ]:
type(lista_cadenas[0])

str

Gracias al tipado dinámico de Python, incluso podemos crear listas heterogéneas:

In [ ]:
lista_mixta = [True, "2", 3.0, 4]
[type(elemento) for elemento in lista_mixta]

[bool, str, float, int]

Pero esta flexibilidad tiene un coste: para permitir estos tipos flexibles, cada elemento de la lista debe contener su propia información de:
- tipo.
- contador de referencias.
- otra información. 

Es decir, cada elemento es un <span style="color:Gold">objeto Python completo</span>. En el caso especial de que todas las variables sean del mismo tipo, gran parte de esta información es redundante: puede ser mucho más eficiente almacenar datos en un array de tipo fijo. La diferencia entre una lista de tipo dinámico y un array de tipo fijo (al estilo NumPy) se ilustra en la siguiente figura:

![Distribución de memoria: array vs lista](https://i.imgur.com/KvfztfZ.png)

<div class="alert alert-block alert-info">
<b>Pregunta:</b> Viendo la imagen. ¿Puedes intuir qué diferencias hay entre una lista de python y un array de NumPy?
</div>

A nivel de implementación, el array contiene esencialmente un único puntero a un bloque contiguo de datos. La lista de Python, por otro lado, contiene un puntero a un bloque de punteros, cada uno de los cuales apunta a su vez a un objeto Python completo como el entero Python que vimos antes. De nuevo, la ventaja de la lista es la flexibilidad: como cada elemento de la lista es una estructura completa que contiene tanto datos como información de tipo, la lista puede llenarse con datos de cualquier tipo deseado. Los arrays de tipo fijo al estilo NumPy carecen de esta flexibilidad, pero son mucho más eficientes para almacenar y manipular datos.